# 02 Rainfall validation against the CHMI rain gauges

`scripts/derive_5min.py` derives the 5-minute rainfall from the released 1-minute file; `scripts/gauge_validation.py`
compares it with the nearest rain gauge within 5 km (Sect. 4 of the paper); `scripts/figures/fig05_fig06_validation.py`
draws Figs. 5 and 6 from the saved samples. The derivation and the validation take several minutes each.

In [ ]:
from pathlib import Path
import json, subprocess, sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from essd_cz.config import load_config

CONFIG = ROOT / 'config' / 'user_paths.json'   # copy of config/user_paths.example.json with your paths
cfg = load_config(CONFIG)
paths, OUT = cfg['datasets'], cfg['output_dir']


def run(script, *args):
    """Run a script of the repository with the configuration of this notebook."""
    cmd = [sys.executable, str(ROOT / script), '--config', str(CONFIG), *map(str, args)]
    print(' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True)

In [ ]:
from essd_cz.rain5 import derived_5min_name
if not (OUT / 'rainfall_5min' / derived_5min_name(paths['rain'].name)).exists():
    run('scripts/derive_5min.py')
if not (OUT / 'validation' / 'graf_summary.json').exists():
    run('scripts/gauge_validation.py')

In [ ]:
summary = json.loads((OUT / 'validation' / 'graf_summary.json').read_text())
for k, v in summary.items():
    if k not in ('inputs', 'method'):
        print(k, json.dumps(v, indent=1)[:1500])

In [ ]:
run('scripts/figures/fig05_fig06_validation.py')
from IPython.display import Image, display
for f in ('FINAL_GRAF_confusion_GAUGE_same_population_202104_202110.png', 'FINAL_GRAF_hexbin_gauge_WETONLY_202104_202110.png'):
    display(Image(filename=str(OUT / 'figures' / 'fig05_fig06' / f), width=500))